# Bayronik: train the N-body → hydro emulator on a Colab GPU

Runtime → Change runtime type → **A100** (or L4/T4; slower). Run the cells top to bottom.
The notebook fetches the CAMELS maps itself (7.9 GB) and saves weights to Google Drive after every epoch, so a disconnect loses at most one epoch: rerun all cells and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code: a bundle uploaded to /content wins; otherwise clone the branch from GitHub.
import os
BRANCH = 'master'
if os.path.exists('/content/bayronik_bundle.tar.gz'):
    !mkdir -p /content/Bayronik/model && tar -xzf /content/bayronik_bundle.tar.gz -C /content/Bayronik/model
elif not os.path.exists('/content/Bayronik'):
    !git clone --depth 1 -b $BRANCH https://github.com/yuvichh/Bayronik.git /content/Bayronik
%cd /content/Bayronik/model
!ls

In [ ]:
# Weights live on Drive so they survive the runtime.
RUN = 'ufno_v2_Mtot_Nbody_IllustrisTNG'
!mkdir -p /content/drive/MyDrive/bayronik_weights/$RUN weights
!ln -sfn /content/drive/MyDrive/bayronik_weights/$RUN weights/$RUN
!ls -la weights/$RUN

In [ ]:
# BUDGET_MIN=0 trains the fixed number of EPOCHS; a positive value fits the schedule to that many minutes.
!RESUME=1 BUDGET_MIN=0 EPOCHS=40 RUN_NAME=$RUN DATA_DIR=/content/data bash scripts/cloud_train.sh

In [ ]:
!tail -5 weights/$RUN/log.jsonl
!cat weights/$RUN/benchmark.md